In [2]:
import chromadb
from pathlib import Path

# Create database folder inside notebooks
Path("chroma_db").mkdir(exist_ok=True)

# Initialize ChromaDB
client = chromadb.PersistentClient(path="chroma_db")

# Create collection
collection = client.get_or_create_collection(
    name="customer_support_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")
print("Collection name:", collection.name)

ChromaDB collection created successfully!
Collection name: customer_support_knowledge


In [3]:
import os

# Display current working directory
print("Current working directory:")
print(os.getcwd())

# Display files and folders
print("\nAvailable files and folders:")
print(os.listdir())

Current working directory:
C:\Users\PRAVEEN\OneDrive\Attachments\Desktop\customer_support_ai\notebooks

Available files and folders:
['.ipynb_checkpoints', 'app.py', 'chroma_db', 'customer_support_ai.ipynb', 'customer_support_rag.ipynb', 'data', 'models', 'notebooks', 'rag_app.py', 'reports', 'Untitled.ipynb']


In [4]:
from pathlib import Path

# Search for the processed chunks CSV
search_location = Path.home()

matches = list(
    search_location.glob(
        "**/customer_support_chunks.csv"
    )
)

for file in matches:
    print("File found:", file)

File found: C:\Users\PRAVEEN\OneDrive\Attachments\Desktop\customer_support_ai\notebooks\data\processed\customer_support_chunks.csv


In [5]:
import pandas as pd

knowledge_base = pd.read_csv(
    r"C:\Users\PRAVEEN\OneDrive\Attachments\Desktop\customer_support_ai\notebooks\data\processed\customer_support_chunks.csv"
)

print("Knowledge base loaded successfully!")
print("Total rows:", len(knowledge_base))
print("Columns:", knowledge_base.columns.tolist())

knowledge_base.head()

Knowledge base loaded successfully!
Total rows: 26872
Columns: ['instruction_raw', 'response', 'intent', 'text_chunk']


,instruction_raw,response,intent,text_chunk
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,Customer Question: question about cancelling o...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,Customer Question: i have a question about can...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,Customer Question: i need help cancelling puch...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,Customer Question: I need to cancel purchase {...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,"Customer Question: I cannot afford this order,..."


In [6]:
from pathlib import Path

search_location = Path.home()

matches = list(
    search_location.glob(
        "**/customer_support_embeddings.npy"
    )
)

for file in matches:
    print("Embeddings file found:", file)

Embeddings file found: C:\Users\PRAVEEN\OneDrive\Attachments\Desktop\customer_support_ai\notebooks\data\processed\customer_support_embeddings.npy


In [7]:
import numpy as np

embeddings = np.load(
    r"C:\Users\PRAVEEN\OneDrive\Attachments\Desktop\customer_support_ai\notebooks\data\processed\customer_support_embeddings.npy"

)

print("Embeddings loaded successfully!")
print("Embedding shape:", embeddings.shape)

Embeddings loaded successfully!
Embedding shape: (26872, 384)


In [8]:
# Prepare documents
documents = knowledge_base["text_chunk"].astype(str).tolist()

# Prepare metadata
metadatas = [
    {
        "intent": str(intent),
        "question": str(question),
        "answer": str(answer)
    }
    for intent, question, answer in zip(
        knowledge_base["intent"],
        knowledge_base["instruction_raw"],
        knowledge_base["response"]
    )
]

# Generate unique IDs
ids = [f"support_{i}" for i in range(len(documents))]

# Insert records in batches
batch_size = 5000

for i in range(0, len(documents), batch_size):
    collection.add(
        ids=ids[i:i + batch_size],
        documents=documents[i:i + batch_size],
        embeddings=embeddings[i:i + batch_size].tolist(),
        metadatas=metadatas[i:i + batch_size]
    )

print("All embeddings stored successfully!")
print("Total records:", collection.count())

All embeddings stored successfully!
Total records: 26872


In [9]:
from sentence_transformers import SentenceTransformer

# Load the saved pretrained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [10]:
# Sample customer question
query = "I want to cancel my order"

# Generate embedding for the query
query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
).tolist()

# Search ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Display search results
for i in range(len(results["documents"][0])):
    print(f"\nResult {i+1}")
    print("Question:", results["metadatas"][0][i]["question"])
    print("Intent:", results["metadatas"][0][i]["intent"])
    print("Answer:", results["metadatas"][0][i]["answer"])


Result 1
Question: would you give me information about cancelling orders?
Intent: cancel_order
Answer: Of course! I'd be happy to provide you with information about canceling orders. Canceling an order can usually be done through your account on our website or by contacting our customer support team. The exact process may vary depending on the platform or service you used to place the order. To cancel an order, follow these general steps:

1. Sign in to Your Account: Access your account on our platform by entering your login credentials.
2. Locate Your Order: Find the order you wish to cancel either in your order history or in the specific section for managing orders.
3. Initiate the Cancellation: Look for the "Cancel" or "Request Cancellation" option associated with the order and click on it.
4. Follow any Prompts or Instructions: Depending on the system, you may need to provide a reason for canceling or confirm your cancellation request.
5. Verify the Cancellation: After completing 

In [11]:
def retrieve_answers(query, top_k=5):

    # Convert customer question into embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Search the vector database
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    # Display retrieved answers
    for i in range(len(results["documents"][0])):

        print(f"\nResult {i+1}")
        print("Question:", results["metadatas"][0][i]["question"])
        print("Intent:", results["metadatas"][0][i]["intent"])
        print("Answer:", results["metadatas"][0][i]["answer"])

    return results


print("Retrieval function created successfully!")

Retrieval function created successfully!


In [12]:
# Test 1
print("CUSTOMER QUESTION 1")
retrieve_answers("I want to cancel my order", top_k=3)

# Test 2
print("\nCUSTOMER QUESTION 2")
retrieve_answers("Where is my package?", top_k=3)

# Test 3
print("\nCUSTOMER QUESTION 3")
retrieve_answers("I want to get my money back", top_k=3)

CUSTOMER QUESTION 1

Result 1
Question: would you give me information about cancelling orders?
Intent: cancel_order
Answer: Of course! I'd be happy to provide you with information about canceling orders. Canceling an order can usually be done through your account on our website or by contacting our customer support team. The exact process may vary depending on the platform or service you used to place the order. To cancel an order, follow these general steps:

1. Sign in to Your Account: Access your account on our platform by entering your login credentials.
2. Locate Your Order: Find the order you wish to cancel either in your order history or in the specific section for managing orders.
3. Initiate the Cancellation: Look for the "Cancel" or "Request Cancellation" option associated with the order and click on it.
4. Follow any Prompts or Instructions: Depending on the system, you may need to provide a reason for canceling or confirm your cancellation request.
5. Verify the Cancellatio

{'ids': [['support_16624', 'support_16546', 'support_16088']],
 'embeddings': None,
 'documents': [["Customer Question: were do i obtain my money back\nCustomer Support Answer: I've got that you are seeking information on where to obtain your money back. Our customer service team will be more than happy to assist you in this matter. You can reach out to them through the Live Chat on our website at {{Website URL}} or by calling {{Customer Support Phone Number}}. They will guide you through the process and ensure that you receive the necessary support to obtain your refund.",
   "Customer Question: getting money back\nCustomer Support Answer: I realized, you're seeking assistance to get your money back, and I'm here to help you navigate through the process. Obtaining a refund can vary depending on the specific circumstances and policies involved. To ensure a prompt resolution, I recommend reaching out to our customer support team who specialize in handling refund requests. They will be a

In [13]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [14]:
import ollama

# Test the Llama model
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "Hello! Introduce yourself as a customer support assistant."
        }
    ]
)

print(response["message"]["content"])

Hello! My name is Rachel, and I'll be your customer support assistant today. I'm here to help answer any questions you may have, troubleshoot any issues you're experiencing, and provide you with solutions to get you back on track.

If you're having trouble with a product or service, please feel free to share more details about what's not working the way you want it to, and I'll do my best to assist you. You can also ask me any general questions you may have, and I'll do my best to provide you with helpful information.

Before we get started, I just want to let you know that I'll be available to chat with you 24/7, and I'll do my best to respond as quickly as possible. If I'm unable to help you right away, I can also escalate your issue to a more senior member of our team for further assistance.

So, what can I help you with today?


In [15]:
def rag_chatbot(query):

    # Retrieve relevant documents
    results = collection.query(
        query_embeddings=[
            embedding_model.encode(
                query,
                normalize_embeddings=True
            ).tolist()
        ],
        n_results=3
    )

    # Prepare context from retrieved answers
    context = "\n\n".join(
        results["documents"][0]
    )

    # Create prompt
    prompt = f"""
    You are a helpful customer support assistant.

    Answer the customer's question using only
    the information provided in the context.

    If the answer is not available in the context,
    say that you don't have enough information.

    Context:
    {context}

    Customer Question:
    {query}

    Provide a clear and helpful answer.
    """

    # Generate response using Llama
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


print("RAG chatbot function created successfully!")

RAG chatbot function created successfully!


In [16]:
# Test the RAG chatbot

question = "I want to cancel my order. How can I do that?"

answer = rag_chatbot(question)

print("Customer Question:")
print(question)

print("\nRAG Chatbot Answer:")
print(answer)

Customer Question:
I want to cancel my order. How can I do that?

RAG Chatbot Answer:
To cancel your order, you can follow the steps I provided earlier. Here's a concise version:

To cancel your order, follow these steps:

1. Sign in to your account on our website.
2. Locate your order in the "Orders" or "Order History" section.
3. Find the specific order you want to cancel and click on the "Cancel" or "Request Cancellation" option.
4. Follow any prompts or instructions provided by the system.
5. Verify that your order has been successfully canceled.

If you need further assistance or have specific questions, please don't hesitate to reach out to our dedicated customer support team. They are available during {{Customer Support Hours}} at {{Customer Support Phone Number}} or through the Live Chat feature on our website at {{Website URL}}.


In [17]:
test_questions = [
    "Where is my package?",
    "I want to get my money back.",
    "How can I return a damaged product?"
]

for question in test_questions:

    print("\n" + "=" * 60)
    print("Customer:", question)

    answer = rag_chatbot(question)

    print("\nChatbot:", answer)


Customer: Where is my package?

Chatbot: We don't have enough information to provide a specific location for your package. However, we can guide you on how to find out where your package is. To check the status of your package and its location, you can visit our website and go to the "Order Tracking" or "Track my Package" section. There, you will be prompted to enter your Tracking Number or Order Number for detailed updates.

Customer: I want to get my money back.

Chatbot: I can relate to the understanding that you're looking for guidance on where to demand a refund for your money. It can be frustrating when you're unsure where to direct your request.

To initiate the process, I recommend reaching out to the company or service provider from whom you made the purchase. Start by contacting their customer support team through their website, email, or phone number. They will be able to provide you with the necessary instructions on how to proceed with your refund request.

Remember to pr

In [18]:
def rag_chatbot(query):

    # Retrieve relevant documents
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    # Prepare context
    context = "\n\n".join(results["documents"][0])

    # Strict RAG prompt
    prompt = f"""
You are a customer support assistant.

Answer the customer's question using ONLY the information
provided in the context.

Important rules:
1. Do not invent company policies.
2. Do not create phone numbers or website addresses.
3. Do not invent refund or return conditions.
4. Do not add information that is not in the context.
5. If the context does not contain the answer, clearly say
   that you do not have enough information.
6. Give a clear and simple answer.

CONTEXT:
{context}

CUSTOMER QUESTION:
{query}

ANSWER:
"""

    # Generate response
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


print("Improved RAG chatbot function created successfully!")


Improved RAG chatbot function created successfully!


In [19]:
test_questions = [
    "Where is my package?",
    "I want to get my money back.",
    "How can I return a damaged product?"
]

for question in test_questions:

    print("\n" + "=" * 60)
    print("Customer:", question)

    answer = rag_chatbot(question)

    print("\nChatbot:", answer)


Customer: Where is my package?

Chatbot: I don't have enough information to provide an answer to your question about where your package is.

Customer: I want to get my money back.

Chatbot: I can relate to the understanding that you're looking for guidance on how to proceed with getting your money back. To initiate the process, I recommend reaching out to the company or service provider from whom you made the purchase. Start by contacting their customer support team through their website or by calling them. They will be able to provide you with the necessary instructions on how to proceed with your refund request.

Customer: How can I return a damaged product?

Chatbot: To initiate the return process for a damaged product, please reach out to our customer support team. They will guide you through the necessary steps, gather the required information, and ensure a smooth return process.


In [20]:
while True:

    user_question = input("\nCustomer: ")

    if user_question.lower() == "exit":
        print("Chatbot: Thank you for contacting customer support!")
        break

    answer = rag_chatbot(user_question)

    print("\nChatbot:", answer)


Customer:  exit


Chatbot: Thank you for contacting customer support!


In [21]:
import re

# Find answers containing placeholders
placeholder_pattern = r"\{\{.*?\}\}"

placeholder_rows = knowledge_base[
    knowledge_base["response"].str.contains(
        placeholder_pattern,
        regex=True,
        na=False
    )
]

print("Total knowledge base records:", len(knowledge_base))
print("Records containing placeholders:", len(placeholder_rows))

# Display examples
placeholder_rows[
    ["instruction_raw", "response"]
].head(10)

Total knowledge base records: 26872
Records containing placeholders: 13006


,instruction_raw,response
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...
5,can you help me cancel order {{Order Number}}?,"Of course, I'm here to assist you in canceling..."
6,"I can no longer afford order {{Order Number}},...",I pick up what you're putting down that you're...
7,I am trying to cancel purchase {{Order Number}},I've understood that you're seeking assistance...
8,I have got to cancel purchase {{Order Number}},I'm sensitive to the fact that you're seeking ...
9,i need help canceling purchase {{Order Number}},I perceive that you're seeking assistance with...


In [22]:

test_queries = [
    "I want to cancel my order",
    "Where is my package?",
    "I want to get my money back",
    "My product arrived damaged"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("CUSTOMER QUESTION:", query)
    print("=" * 80)

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )

    for i, doc in enumerate(results["documents"][0]):

        print(f"\nResult {i + 1}")
        print("Distance:", results["distances"][0][i])
        print("Document:")
        print(doc[:700])
        print("-" * 60)



CUSTOMER QUESTION: I want to cancel my order

Result 1
Distance: 0.21229875087738037
Document:
Customer Question: would you give me information about cancelling orders?
Customer Support Answer: Of course! I'd be happy to provide you with information about canceling orders. Canceling an order can usually be done through your account on our website or by contacting our customer support team. The exact process may vary depending on the platform or service you used to place the order. To cancel an order, follow these general steps:

1. Sign in to Your Account: Access your account on our platform by entering your login credentials.
2. Locate Your Order: Find the order you wish to cancel either in your order history or in the specific section for managing orders.
3. Initiate the Cancellatio
------------------------------------------------------------

Result 2
Distance: 0.21290993690490723
Document:
Customer Question: cancel order
Customer Support Answer: I've realized that you're seeking a

In [23]:

# Improved retrieval testing

test_queries = {
    "Cancellation": [
        "Where can I cancel my order?",
        "How do I cancel an existing order?",
        "I no longer want my order."
    ],

    "Package Tracking": [
        "Where is my package?",
        "How can I track my order?",
        "What is the estimated delivery date of my package?",
        "How do I check my shipment status?"
    ],

    "Refund": [
        "How can I request a refund?",
        "How do I get my money back?",
        "How can I return my money?"
    ],

    "Damaged Product": [
        "My product arrived damaged.",
        "I received a broken item.",
        "How can I return a defective product?"
    ]
}

for category, queries in test_queries.items():

    print("\n" + "=" * 80)
    print("CATEGORY:", category)
    print("=" * 80)

    for query in queries:

        query_embedding = embedding_model.encode(
            query
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=3,
            include=["documents", "distances"]
        )

        print("\nCustomer Question:", query)

        for i, doc in enumerate(results["documents"][0]):

            print(f"\nResult {i + 1}")
            print("Distance:", round(
                results["distances"][0][i], 4
            ))

            print(doc[:350])
            print("-" * 50)



CATEGORY: Cancellation

Customer Question: Where can I cancel my order?

Result 1
Distance: 0.2425
Customer Question: where could I cancel purchase {{Order Number}}?
Customer Support Answer: I've realized you're looking for the location where you can cancel your purchase with order number {{Order Number}}. Our cancellation process can be conveniently completed through our online platform. Here's what you need to do:

1. Visit our {{Website URL}}
--------------------------------------------------

Result 2
Distance: 0.2605
Customer Question: would you give me information about cancelling orders?
Customer Support Answer: Of course! I'd be happy to provide you with information about canceling orders. Canceling an order can usually be done through your account on our website or by contacting our customer support team. The exact process may vary depending on the platform
--------------------------------------------------

Result 3
Distance: 0.2694
Customer Question: where can I cancel orde

In [24]:

# Query Expansion for Customer Support RAG

def improve_search_query(query):

    query_lower = query.lower()

    if any(word in query_lower for word in [
        "package",
        "shipment",
        "delivery status",
        "where is my order"
    ]):

        return "How can I track my order and check its shipment status and estimated delivery date?"

    elif any(word in query_lower for word in [
        "cancel",
        "cancellation"
    ]):

        return "How can I cancel an existing order? Provide order cancellation instructions."

    elif any(word in query_lower for word in [
        "refund",
        "money back",
        "reimbursement"
    ]):

        return "How can I request a refund and get my money back?"

    elif any(word in query_lower for word in [
        "damaged",
        "broken",
        "defective",
        "faulty"
    ]):

        return "I received a damaged or defective product. How can I report the issue and request a return?"

    else:

        return query


# Test Query Expansion

test_questions = [
    "Where is my package?",
    "I want to cancel my order",
    "I want to get my money back",
    "My product arrived damaged"
]

for question in test_questions:

    improved_query = improve_search_query(question)

    print("Original:", question)
    print("Improved:", improved_query)
    print("-" * 60)


Original: Where is my package?
Improved: How can I track my order and check its shipment status and estimated delivery date?
------------------------------------------------------------
Original: I want to cancel my order
Improved: How can I cancel an existing order? Provide order cancellation instructions.
------------------------------------------------------------
Original: I want to get my money back
Improved: How can I request a refund and get my money back?
------------------------------------------------------------
Original: My product arrived damaged
Improved: I received a damaged or defective product. How can I report the issue and request a return?
------------------------------------------------------------


In [27]:

# Test retrieval using improved search queries

test_questions = [
    "Where is my package?",
    "I want to cancel my order",
    "I want to get my money back",
    "My product arrived damaged"
]

for question in test_questions:

    improved_query = improve_search_query(question)

    query_embedding = embedding_model.encode(
        improved_query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "distances"]
    )

    print("\n" + "=" * 80)
    print("ORIGINAL QUESTION:", question)
    print("IMPROVED QUERY:", improved_query)
    print("=" * 80)

    for i, doc in enumerate(results["documents"][0]):

        print(f"\nResult {i + 1}")
        print(
            "Distance:",
            round(results["distances"][0][i], 4)
        )
        print(doc[:500])
        print("-" * 50)



ORIGINAL QUESTION: Where is my package?
IMPROVED QUERY: How can I track my order and check its shipment status and estimated delivery date?

Result 1
Distance: 0.1793
Customer Question: how do I check when my shipment is going to arrive?
Customer Support Answer: We can definitely assist you with tracking the arrival of your shipment. To check the estimated delivery time, you will need to have either your {{Tracking Number}} or {{Order Number}}. Once you have that information, you can visit our website and go to the "Track Order" or "Order Status" page. There, you can enter your tracking number or order number, and it will provide you with the latest updates o
--------------------------------------------------

Result 2
Distance: 0.2162
Customer Question: need to check wwhen my shipment is gonna arrive can uhelp me
Customer Support Answer: We understand your eagerness to track the progress of your shipment and find out when it will arrive. To assist you better, could you please provide

In [28]:
import pandas as pd

df = pd.read_csv(
    "data/processed/customer_support_knowledge_base.csv"
)

In [29]:
refund_data = df[
    df["instruction_raw"].str.contains(
        "refund",
        case=False,
        na=False
    )
]

print(
    refund_data[
        ["instruction_raw", "response"]
    ].head(10).to_string(index=False)
)

                                             instruction_raw                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

## Final Chatbot Testing and Evaluation

The Customer Support AI Chatbot was tested using nine different customer queries covering greetings, order cancellation, order tracking, refunds, damaged products, and support contact information.

**Testing Results:**
- Total test cases: 9
- Passed: 9
- Partial: 0
- Failed: 0

**Conclusion:**

The chatbot successfully handled all nine functional test scenarios. It provided relevant customer-support instructions, handled missing information safely, and avoided inventing refund processing times, support phone numbers, or replacement policies.

The chatbot is ready for the next project phase: documentation and reporting.

In [30]:

import pandas as pd
import numpy as np

# Load customer support data
df = pd.read_csv("data/processed/customer_support_chunks.csv")

# Load embeddings
embeddings = np.load("data/processed/customer_support_embeddings.npy")

# Display results
print("CSV shape:", df.shape)
print("Embeddings shape:", embeddings.shape)

print("\nCSV columns:")
print(df.columns.tolist())

print("\nFirst 3 records:")
display(df.head(3))


CSV shape: (26872, 4)
Embeddings shape: (26872, 384)

CSV columns:
['instruction_raw', 'response', 'intent', 'text_chunk']

First 3 records:


,instruction_raw,response,intent,text_chunk
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,Customer Question: question about cancelling o...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,Customer Question: i have a question about can...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,Customer Question: i need help cancelling puch...
